# Building the Model - Fantasy Hockey Weekly Predictions

This notebook builds and explains the base statistical model (v1) used for
weekly predictions in this project. It's meant to be read, not just run,
each section explains why a choice was made, not just what the code does.

What this model does: estimates a player's expected fantasy performance and
how consistent that performance is, then uses both to compute a probability
that one player (or goalie) will outperform another in a given week.

What this model does not do: predict which team wins an actual NHL game.
See METHODOLOGY.md in the repo for the full write-up.

In [ ]:
#Main packages used
import numpy as np
import pandas as pd
from scipy.stats import norm

import json
from google.colab import files

import requests
import time

import os
from datetime import datetime, timedelta

import random


pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", None)

## Scoring configuration

Fantasy stat categories differ for skaters (forwards/defensemen) vs.
goaltenders, and a custom scoring upload should only be allowed to use
recognized categories - otherwise a typo or unsupported stat would
silently produce wrong fantasy point totals downstream with no warning.

Below, the default scoring config is set to my own league's actual rules.
An optional custom upload is checked against a fixed list of valid Yahoo
fantasy hockey stat abbreviations (split into skater and goalie categories)
before it's allowed to override anything. Any unrecognized key is flagged
and dropped rather than silently accepted. The abbreviations used here are:

**Player Stats**
*   **G**: Goals
*   **A**: Assists
*   **P**: Points
*   **+/-**: Plus/Minus Rating
*   **PIM**: Penalty Minutes
*   **PPG**: Powerplay Goals
*   **PPA**: Powerplay Assists
*   **PPP**: Powerplay Points
*   **SHG**: Shorthanded Goals
*   **SHA**: Shorthanded Assists
*   **SHP**: Shorthanded Points
*   **GWG**: Game-Winning Goals
*   **SOG**: Shots on Goal
*   **SH%**: Shooting Percentage
*   **FW**: Faceoffs Won
*   **FL**: Faceoffs Lost
*   **HIT**: Hits
*   **BLK**: Blocks

**Goalie Stats**
*   **GS**: Games Started
*   **W**: Wins
*   **L**: Losses
*   **SHO**: Shutouts
*   **SA**: Shots Against
*   **SV**: Saves
*   **GA**: Goals Against
*   **GAA**: Goals Against Average
*   **SV%**: Save Percentage


*Note: FW, FL are not currently scoreable, only a faceoff percentage is
available from this data source, not raw win/loss counts, see
future_directions for details.*



In [ ]:
# Recognized fantasy stat categories (skaters and goalies use different sets).
# Based on standard Yahoo Fantasy Hockey abbreviations.

VALID_SKATER_STATS = {
    "G", "A", "P", "+/-", "PIM", "PPG", "PPA", "PPP",
    "SHG", "SHA", "SHP", "GWG", "SOG", "SH%", "FW", "FL", "HIT", "BLK"
}

VALID_GOALIE_STATS = {
    "GS", "W", "L", "SHO", "SA", "SV", "GA", "GAA", "SV%"
}

# My "main" league's actual scoring - used as the default.
default_skater_scoring = {
    "G": 2,
    "A": 1,
    "PIM": -0.5,
    "PPG": 0.5,
    "SHG": 2,
    "SHA": 1,
    "GWG": 1,
    "SOG": 0.1,
    "HIT": 0.25,
    "BLK": 0.25,
}

default_goalie_scoring = {
    "W": 5,
    "GA": -1,
    "SV": 0.1,
    "SHO": 5,
}

skater_scoring = default_skater_scoring
goalie_scoring = default_goalie_scoring

In [ ]:
def validate_scoring_config(config, valid_stats, label):
    """
    Checks that every key in a scoring config is a recognized fantasy stat.
    Returns (clean_config, rejected_keys) - invalid keys are dropped, not silently kept.
    """
    clean_config = {}
    rejected = []

    for stat, value in config.items():
        if stat in valid_stats:
            clean_config[stat] = value
        else:
            rejected.append(stat)

    if rejected:
        print(f"Warning: the following {label} keys were not recognized and were dropped: {rejected}")

    return clean_config, rejected

# Sanity check against our own defaults, should show zero rejections
skater_scoring, _ = validate_scoring_config(skater_scoring, VALID_SKATER_STATS, "skater")
goalie_scoring, _ = validate_scoring_config(goalie_scoring, VALID_GOALIE_STATS, "goalie")

print("Skater scoring:", skater_scoring)
print("Goalie scoring:", goalie_scoring)

In [ ]:
#If you want to import a custom player scoring file.
print("Upload a custom skater scoring config (JSON), or skip this cell to keep the default.")
uploaded = files.upload()

if uploaded:
    custom_filename = list(uploaded.keys())[0]
    with open(custom_filename) as f:
        candidate_config = json.load(f)
    skater_scoring, rejected = validate_scoring_config(candidate_config, VALID_SKATER_STATS, "skater")
    print(f"Custom skater scoring loaded from {custom_filename}: {skater_scoring}")
print("Skater scoring:", skater_scoring)

In [ ]:
#If you want to import a custom goalie scoring file.
print("Upload a custom goalie scoring config (JSON), or skip this cell to keep the default.")
uploaded = files.upload()

if uploaded:
    custom_filename = list(uploaded.keys())[0]
    with open(custom_filename) as f:
        candidate_goalie_config = json.load(f)
    goalie_scoring, rejected = validate_scoring_config(candidate_goalie_config, VALID_GOALIE_STATS, "goalie")
    print(f"Custom goalie scoring loaded from {custom_filename}: {goalie_scoring}")
print("Goalie scoring:", goalie_scoring)

## Section 1, getting raw per-game data

The model needs a per-game stat line for each player, goals, assists,
shots, and so on. In the live version of this project, this comes from
the NHL's public API. For this notebook, I'm using a small hand-built
sample dataset for two players so the logic below is easy to follow and
reproduce without needing a live connection, the actual data pulling
functions get built out later in this notebook.

The two example players below are deliberately chosen to illustrate the
point of this whole model, Player A is a steady, consistent scorer,
Player B has the same average production, but is far more boom-or-bust.
A model that only looked at averages would treat them as identical.

## Where the data comes from

Stats in this project come from the NHL's public web API
(api-web.nhle.com), specifically the player game log endpoint. It's
free, official, and doesn't need an API key, which matters since this
whole project is meant to run on open data anyone can access.

How far back it goes, technically the API has structured data going back
decades, even to the early 1900s for basic schedule info. But detailed,
game by game player stats are really only reliable from around the mid
2000s onward, once the league started tracking things more consistently.

For this model though, we don't need most of that history. What we
actually pull is the last season or two, used as the prior for the
shrinkage estimate, plus the current season, updated week to week as
games are played. So in practice, this notebook will mostly be working
with the last 1 to 2 seasons of data, not the full historical archive.
That's intentional, older data is less relevant to how a player is
performing right now anyway.

One thing worth flagging honestly, this is an unofficial, undocumented
API in the sense that the NHL doesn't publish a formal spec for it. It's
widely used by hobby projects and has been stable for a while, but it
could change or break without notice. If that happens, the fix is just
updating the pull functions in this notebook, the rest of the model
doesn't care where the numbers come from.

## Section 2, building a full player roster

Rather than looking players up one at a time by name, it's more useful
to pull a complete table of every active player, across all 32 teams,
once. That gives us player ids, names, positions, and teams all in one
place, which is what we need for things like scanning a whole roster,
or later, finding sleeper picks league wide.

Goalies show up in this same roster pull, but worth flagging now, once
we get to actual game logs, goalies need separate handling. Their stat
categories are completely different, saves, goals against, decisions,
not goals and assists, so a goalie's game log comes back from the API
with different fields entirely. We'll build a separate goalie pull
function for that reason, not just filter the skater one.

In [ ]:
NHL_TEAMS = [
    "ANA","BOS","BUF","CAR","CBJ","CGY","CHI","COL","DAL","DET",
    "EDM","FLA","LAK","MIN","MTL","NJD","NSH","NYI","NYR","OTT",
    "PHI","PIT","SEA","SJS","STL","TBL","TOR","UTA","VAN","VGK",
    "WPG","WSH"
]

def get_team_roster(team_abbr, retries=3, delay=1):
    """
    Pulls the current roster for one team, returns a DataFrame with
    id, name, and position for every player, skaters and goalies together.
    Retries a couple times with a growing delay if rate limited.
    """
    url = f"https://api-web.nhle.com/v1/roster/{team_abbr}/current"

    for attempt in range(retries):
        response = requests.get(url)
        if response.status_code == 429:
            wait = delay * (attempt + 1)
            print(f"Rate limited on {team_abbr}, waiting {wait}s and retrying")
            time.sleep(wait)
            continue
        response.raise_for_status()
        data = response.json()

        rows = []
        for group in ["forwards", "defensemen", "goalies"]:
            for player in data.get(group, []):
                rows.append({
                    "player_id": player["id"],
                    "name": f"{player['firstName']['default']} {player['lastName']['default']}",
                    "position": player["positionCode"],
                    "team": team_abbr,
                })
        return pd.DataFrame(rows)

    raise RuntimeError(f"Failed to pull roster for {team_abbr} after {retries} attempts")

def get_all_players(pause=0.5):
    """
    Loops over every team and builds one big table of all active players.
    Pauses briefly between calls to stay under the API's rate limit.
    """
    all_rosters = []
    for team in NHL_TEAMS:
        roster = get_team_roster(team)
        all_rosters.append(roster)
        time.sleep(pause)
    return pd.concat(all_rosters, ignore_index=True)

all_players = get_all_players()
all_players.tail()


## Section 3, checking the roster pull actually worked

Before trusting all_players for anything downstream, worth running a
few basic checks, not a full test suite, just enough to catch obvious
problems, like a team silently failing, or goalies missing entirely ☹.

In [ ]:
def check_roster_data(players_df):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    # basic shape
    check(len(players_df) > 0, "players_df is not empty")
    check(set(["player_id", "name", "position", "team"]).issubset(players_df.columns),
          "has the expected columns")

    # every team is represented, nothing silently dropped
    missing_teams = set(NHL_TEAMS) - set(players_df["team"].unique())
    check(len(missing_teams) == 0, f"all {len(NHL_TEAMS)} teams are present")
    if missing_teams:
        print(f"       missing teams: {missing_teams}")

    # goalies specifically made it in
    goalie_count = (players_df["position"] == "G").sum()
    check(goalie_count > 0, f"goalies are present ({goalie_count} found)")
    check(goalie_count >= 32 * 2, "roughly the expected number of goalies (at least 2 per team)")

    # no duplicate player ids, would suggest something got pulled twice
    check(players_df["player_id"].is_unique, "no duplicate player ids")

    # rough sanity check on total size, NHL rosters are usually 20-23 active players
    expected_min = 32 * 30
    expected_max = 32 * 55
    check(expected_min <= len(players_df) <= expected_max,
          f"total player count ({len(players_df)}) is in a sane range")

    print(f"\n{checks_passed}/{checks_total} checks passed")

check_roster_data(all_players)

## Section 4, caching and cleaning

Two different caching needs here. The player roster barely changes day
to day, so it just needs an occasional refresh, once a week is plenty.
Game logs are different, during the season they change every time a
game is played, so those need to be checked more often, but still not
refetched on literally every run of the notebook.

The rule, roughly, if the cached file is older than some threshold,
pull fresh data and overwrite it, otherwise just read what's already
saved. Past season data isn't handled yet, that comes back once we get
to the shrinkage section, since that's the first place it's actually
needed.

In [ ]:
def fetch_game_log(player_id, season, game_type=2):
    """
    Pulls a player's game log for a given season.
    season format is like 20232024, game_type 2 is regular season, 3 is playoffs.
    Returns a DataFrame, one row per game.
    """
    url = f"https://api-web.nhle.com/v1/player/{player_id}/game-log/{season}/{game_type}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    games = data.get("gameLog", [])
    return pd.DataFrame(games)

In [ ]:
def get_current_season():
    """
    Works out the current NHL season string, like 20262027, based on today's date.
    New seasons typically start in October, so anything July or later counts as
    the start of a new season year.
    """
    today = datetime.now()
    start_year = today.year if today.month >= 7 else today.year - 1
    return f"{start_year}{start_year + 1}"

CURRENT_SEASON = get_current_season()
CURRENT_SEASON
LAST_SEASON = "20252026"

In [ ]:
def get_game_log(player_id, season=CURRENT_SEASON, game_type=2, refresh_after_hours=6, force_refresh=False):
    key = f"gamelog_{player_id}_{season}"
    age = cache_age_hours(key)

    if not force_refresh and age is not None and age < refresh_after_hours:
        print(f"Loading game log for {player_id}, {season} from cache, {age:.1f} hours old")
        return load_from_cache(key)

    print(f"Pulling fresh game log for {player_id}, {season}")
    log_df = fetch_game_log(player_id, season, game_type)

    if log_df.empty:
        if season == CURRENT_SEASON:
            print(f"No games found for {player_id} in {season}, season may not have started yet")
        else:
            print(f"No games found for {player_id} in {season}, player likely wasn't in the NHL that season")
        return log_df

    save_to_cache(log_df, key)
    return log_df

In [ ]:
CACHE_DIR = "data"
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(key):
    return os.path.join(CACHE_DIR, f"{key}.csv")

def cache_age_hours(key):
    """Returns how old a cached file is, in hours, or None if it doesn't exist yet."""
    path = cache_path(key)
    if not os.path.exists(path):
        return None
    modified = datetime.fromtimestamp(os.path.getmtime(path))
    return (datetime.now() - modified).total_seconds() / 3600

def save_to_cache(df, key):
    if df.empty:
        print(f"Not caching '{key}', the data is empty (nothing to save yet)")
        return
    df.to_csv(cache_path(key), index=False)

def load_from_cache(key):
    return pd.read_csv(cache_path(key))

In [ ]:
def get_current_players(refresh_after_hours=168, force_refresh=False):
    """
    Loads the full player roster, from cache if it's fresh enough,
    otherwise pulls fresh and re-caches. Default refresh window is a
    week (168 hours), since rosters don't change often.
    """
    age = cache_age_hours("all_players")

    if not force_refresh and age is not None and age < refresh_after_hours:
        print(f"Loading all_players from cache, {age:.1f} hours old")
        return load_from_cache("all_players")

    print("Pulling fresh player roster data")
    players_df = get_all_players()
    save_to_cache(players_df, "all_players")
    return players_df

all_players = get_current_players()

In [ ]:
def lookup_player_id(name, players_df=all_players):
    """
    Looks up a player's id from the all_players table by name.
    Simple substring match, case insensitive, so partial names work too.
    """
    matches = players_df[players_df["name"].str.contains(name, case=False)]

    if matches.empty:
        print(f"No player found matching '{name}'")
        return None

    if len(matches) > 1:
        print(f"Multiple matches for '{name}', returning the first one:")
        print(matches[["name", "team", "position"]])

    return matches.iloc[0]["player_id"]

## Cleaning the game log

A few things worth fixing before this data feeds into any stats,
missing values, in case a game came back incomplete, columns that
should be numbers but arrive as text, and turning the date column into
an actual date, mostly so we can sort correctly and check "how old is
this data" downstream.

In [ ]:
def clean_game_log(log_df, numeric_columns, required_columns=None):
    """
    Cleans a raw game log. numeric_columns get coerced to numeric types.
    required_columns (defaults to numeric_columns if not given) get checked
    for missing values, but aren't forced to be numeric, this matters for
    columns like 'decision', which are categorical, not numbers.
    """
    df = log_df.copy()

    if required_columns is None:
        required_columns = numeric_columns

    if "gameDate" in df.columns:
        df["gameDate"] = pd.to_datetime(df["gameDate"])

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=[c for c in required_columns if c in df.columns])
    dropped = before - len(df)
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing stats")

    return df.sort_values("gameDate").reset_index(drop=True) if "gameDate" in df.columns else df

## Section 5, checking the full pipeline works, skaters and goalies both

Before trusting this for anything real, worth checking pull, clean, and
cache all work correctly, and specifically that goalies come through
with their own stat fields, not just skater fields with blanks. Same
approach as the roster checks earlier, a handful of pass/fail checks,
not a full test suite.

In [ ]:
GOALIE_NUMERIC_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]
GOALIE_REQUIRED_COLUMNS = ["decision", "savePctg", "goalsAgainst", "shotsAgainst"]

def check_game_log_pipeline(player_id, player_label, stat_columns, season=CURRENT_SEASON, required_columns=None):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Checking pipeline for {player_label} (id: {player_id}) ---")

    raw_log = get_game_log(player_id, season=season, force_refresh=True)
    check(len(raw_log) > 0, "raw game log is not empty")

    all_expected_cols = required_columns if required_columns is not None else stat_columns
    check(all(col in raw_log.columns for col in all_expected_cols),
          f"expected stat columns are present ({all_expected_cols})")

    cleaned_log = clean_game_log(raw_log, stat_columns, required_columns=required_columns)
    check(len(cleaned_log) > 0, "cleaned game log is not empty after cleaning")
    check(len(cleaned_log) <= len(raw_log), "cleaning didn't somehow add rows")

    for col in stat_columns:
        if col in cleaned_log.columns:
            check(pd.api.types.is_numeric_dtype(cleaned_log[col]) or cleaned_log[col].dtype == object,
                  f"'{col}' has a sane dtype after cleaning")

    if "gameDate" in cleaned_log.columns:
        check(cleaned_log["gameDate"].is_monotonic_increasing, "games are sorted by date")

    check(not cleaned_log.duplicated(subset=["gameDate"]).any() if "gameDate" in cleaned_log.columns else True,
          "no duplicate games in the log")

    cached_log = get_game_log(player_id, season=season, force_refresh=False)
    age = cache_age_hours(f"gamelog_{player_id}_{season}")
    check(age is not None and age < 0.1, "second pull loaded from cache, not a fresh request")
    check(len(cached_log) == len(raw_log), "cached data matches what was originally pulled")

    print(f"{checks_passed}/{checks_total} checks passed for {player_label}")
    return checks_passed == checks_total

Section 5 recap, a few real issues turned up while testing this end to
end, an empty pre-season pull crashing the cache, a cleaning step that
was silently dropping every goalie's data regardless of how many games
they'd played, and a stat column list that only covered a fraction of
what the league actually scores. All fixed and verified above. Good
reminder that testing against edge cases, an empty season, a goalie
with one appearance, is what actually catches this kind of thing,
testing only against a star player having a normal season wouldn't
have surfaced any of it.

## Section 6, mapping scoring abbreviations to actual API fields

The scoring config uses short codes, G, A, SOG, and so on, since that's
how leagues actually describe their rules. The data coming back from
the API uses full field names, goals, assists, shots. Something needs
to sit between the two.

Worth being upfront, not every abbreviation maps directly to a field.
Some are a clean 1 to 1, some need a small calculation, like turning a
decision letter into a win or loss, or saves into shotsAgainst minus
goalsAgainst. A couple, hits and blocks specifically, aren't in this
endpoint at all, those would need a different data source entirely,
already noted in future_directions.

In [ ]:
SKATER_FIELD_MAP = {
    "G":    "goals",
    "A":    "assists",
    "P":    "points",
    "+/-":  "plusMinus",
    "PIM":  "pim",
    "PPG":  "powerPlayGoals",
    "PPA":  "ppAssists",       # derived, powerPlayPoints minus powerPlayGoals
    "PPP":  "powerPlayPoints",
    "SHG":  "shorthandedGoals",
    "SHA":  "shAssists",        # derived, shorthandedPoints minus shorthandedGoals
    "SHP":  "shorthandedPoints",
    "GWG":  "gameWinningGoals",
    "SOG":  "shots",
    "SH%":  "shootingPctg",
    "HIT":  "hits",              # from boxscore
    "BLK":  "blockedShots",      # from boxscore
    "FW":   None,   # not available as a raw count, only a percentage exists
    "FL":   None,   # same issue
}

GOALIE_FIELD_MAP = {
    "SA":   "shotsAgainst",
    "SV":   None,   # derived, shotsAgainst minus goalsAgainst
    "GA":   "goalsAgainst",
    "SV%":  "savePctg",
    "SHO":  "shutouts",
    "W":    "decision",
    "L":    "decision",
    "GS":   "gamesStarted",
    "GAA":  None,   # rate stat across games, not meaningful per single game row
}

In [ ]:
GOALIE_NUMERIC_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]
GOALIE_REQUIRED_COLUMNS = ["savePctg", "goalsAgainst", "shotsAgainst"]  # decision no longer required

def clean_goalie_log(log_df, numeric_columns):
    """
    Cleaning specifically for goalie game logs. Never requires 'decision'
    to be present, relief appearances legitimately have none, but still
    have real, scoreable stats. Missing decision becomes 'ND' rather
    than causing the row to be dropped.
    """
    df = clean_game_log(log_df, numeric_columns, required_columns=numeric_columns)
    df["decision"] = df["decision"].fillna("ND") if "decision" in df.columns else "ND"
    return df

In [ ]:
def apply_scoring(games_df, scoring_config, field_map):
    """
    Converts a scoring config, abbreviation based, into fantasy points per game,
    using field_map to translate abbreviations into actual column names.
    Handles the derived cases, W, L, SV, separately since they're not a
    direct column read.
    """
    points = pd.Series(0.0, index=games_df.index)
    unmapped = []

    for stat, weight in scoring_config.items():
        field = field_map.get(stat)

        if stat == "W":
            points += (games_df["decision"] == "W").astype(int) * weight
        elif stat == "L":
            points += (games_df["decision"] == "L").astype(int) * weight
        elif stat == "SV":
            points += (games_df["shotsAgainst"] - games_df["goalsAgainst"]) * weight
        elif field is None:
            unmapped.append(stat)
        else:
            points += games_df[field] * weight

    if unmapped:
        print(f"Warning, these scored categories aren't available from this data source yet, "
              f"and were skipped: {unmapped}")

    return points

In [ ]:
def get_relevant_columns(scoring_config, field_map):
    """
    Works out which raw columns are actually needed to score a given
    scoring config, based on field_map. Ties testing and cleaning
    directly to the league's real scoring rules, so nothing gets missed
    if the scoring config changes later.
    """
    columns = set()
    for stat in scoring_config:
        if stat in ("W", "L"):
            columns.add("decision")
        elif stat == "SV":
            columns.add("shotsAgainst")
            columns.add("goalsAgainst")
        else:
            field = field_map.get(stat)
            if field is not None:
                columns.add(field)
    return sorted(columns)

GOALIE_REQUIRED_COLUMNS = get_relevant_columns(goalie_scoring, GOALIE_FIELD_MAP)

# savePctg isn't always a directly scored category, but it's used constantly
# throughout later analysis (Section 10 especially), so it's always included
# here regardless of what a given league's scoring config happens to score.
GOALIE_NUMERIC_COLUMNS = list(set([c for c in GOALIE_REQUIRED_COLUMNS if c != "decision"] + ["savePctg"]))

print("Goalie required columns:", GOALIE_REQUIRED_COLUMNS)
print("Goalie numeric columns:", GOALIE_NUMERIC_COLUMNS)

## Section 6.5, building the full skater stat line

Turns out a few categories don't need a new data source at all, they're
derivable from fields the game log already gives us. Points already
splits into goals and assists, so the same logic works for powerplay
and shorthanded points, subtract the goals from the points and what's
left is assists in that situation.

Hits and blocked shots do need the boxscore pull from before.

Faceoffs won and lost are a real gap, the API only exposes a faceoff
win percentage, not raw counts, and there's no way to reconstruct wins
and losses from a percentage alone without knowing total faceoffs
taken. Flagged honestly in future_directions rather than faked.

In [ ]:
def get_boxscore(game_id, refresh_after_hours=24, force_refresh=False):
    """
    Pulls the boxscore for a single game, cached by game id. A finished
    game's boxscore never changes, so once cached, it's cached for good,
    the refresh window here mostly matters for a game that's still live.
    """
    key = f"boxscore_{game_id}"
    age = cache_age_hours(key)

    if not force_refresh and age is not None and age < refresh_after_hours:
        return load_from_cache(key)

    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    # flatten playerByGameStats into one row per player for this game
    rows = []
    stats = data.get("playerByGameStats", {})
    for side in ["awayTeam", "homeTeam"]:
        for group in ["forwards", "defense", "goalies"]:
            for player in stats.get(side, {}).get(group, []):
                row = dict(player)
                row["gameId"] = game_id
                rows.append(row)

    boxscore_df = pd.DataFrame(rows)
    if not boxscore_df.empty:
        save_to_cache(boxscore_df, key)
    return boxscore_df

In [ ]:
def get_hits_and_blocks(player_id, game_log, pause=0.3):
    """
    For every game in a player's game log, pulls that game's boxscore
    (cached, reused for other players later) and extracts hits and
    blockedShots for this specific player. Returns a small DataFrame
    keyed by gameId, ready to merge onto the existing game log.
    """
    records = []

    for game_id in game_log["gameId"]:
        boxscore = get_boxscore(game_id)
        if boxscore.empty:
            continue

        player_row = boxscore[boxscore["playerId"] == player_id]
        if player_row.empty:
            continue

        records.append({
            "gameId": game_id,
            "hits": player_row.iloc[0].get("hits", 0),
            "blockedShots": player_row.iloc[0].get("blockedShots", 0),
        })
        time.sleep(pause)

    return pd.DataFrame(records)

In [ ]:
ALL_SKATER_RAW_FIELDS = [
    "goals", "assists", "points", "plusMinus", "pim",
    "powerPlayGoals", "powerPlayPoints",
    "shorthandedGoals", "shorthandedPoints",
    "gameWinningGoals", "shots", "shootingPctg",
]

def build_full_skater_log(player_id, season, include_hits_blocks=True):
    """
    Pulls a skater's game log and builds every stat category we can
    actually source, direct fields, derived fields, and hits/blocks
    from the boxscore. Returns one row per game with everything filled in.
    """
    log = get_game_log(player_id, season=season, force_refresh=False)

    if log.empty:
        return log  # hasn't played this season, nothing to build

    log = clean_game_log(log, ALL_SKATER_RAW_FIELDS)

    # derived categories, no extra api calls needed
    log["ppAssists"] = log["powerPlayPoints"] - log["powerPlayGoals"]
    log["shAssists"] = log["shorthandedPoints"] - log["shorthandedGoals"]

    if include_hits_blocks:
        hits_blocks = get_hits_and_blocks(player_id, log)
        log = log.merge(hits_blocks, on="gameId", how="left")

    return log

In [ ]:
skater_id = lookup_player_id("McDavid")
weegar_id = lookup_player_id("Weegar")
mcdavid_full = build_full_skater_log(skater_id, LAST_SEASON)
weegar_full = build_full_skater_log(weegar_id, LAST_SEASON)

mcdavid_full["fantasy_points"] = apply_scoring(mcdavid_full, skater_scoring, SKATER_FIELD_MAP)
weegar_full["fantasy_points"] = apply_scoring(weegar_full, skater_scoring, SKATER_FIELD_MAP)

print(f"McDavid, avg fantasy points: {mcdavid_full['fantasy_points'].mean():.2f}")
print(f"Mack Daddy, avg fantasy points: {weegar_full['fantasy_points'].mean():.2f}")

## Section 6.6, verifying scoring against known season totals

Two kinds of checks here. First, internal consistency, does points
actually equal goals plus assists in our own data, do the derived
powerplay and shorthanded assist numbers add back up correctly.
Second, external validation, do our summed totals for the season match
McDavid's actual, publicly known 2025-26 stats, 82 games, 48 goals, 90
assists, 138 points, +17, 44 PIM, 306 shots, 13 powerplay goals, 4
game-winning goals. If our pulled data doesn't match these, something
in the pull or cleaning step is wrong, not just the scoring math.

In [ ]:
def check_internal_consistency(log_df, label):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Internal consistency, {label} ---")

    check((log_df["goals"] + log_df["assists"] == log_df["points"]).all(),
          "goals plus assists equals points, every game")

    check((log_df["ppAssists"] + log_df["powerPlayGoals"] == log_df["powerPlayPoints"]).all(),
          "derived ppAssists plus powerPlayGoals equals powerPlayPoints, every game")

    check((log_df["shAssists"] + log_df["shorthandedGoals"] == log_df["shorthandedPoints"]).all(),
          "derived shAssists plus shorthandedGoals equals shorthandedPoints, every game")

    check((log_df["ppAssists"] >= 0).all(), "no negative derived powerplay assists")
    check((log_df["shAssists"] >= 0).all(), "no negative derived shorthanded assists")

    if "hits" in log_df.columns:
        check(log_df["hits"].notna().sum() > 0, "hits data actually came through from boxscore, not all null")
    if "blockedShots" in log_df.columns:
        check(log_df["blockedShots"].notna().sum() > 0, "blocked shots data actually came through, not all null")

    print(f"{checks_passed}/{checks_total} checks passed")

check_internal_consistency(mcdavid_full, "McDavid")

In [ ]:
def check_against_known_totals(log_df, known_totals, label):
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    print(f"\n--- Checking against known real-world totals, {label} ---")

    for stat, expected in known_totals.items():
        actual = log_df[stat].sum()
        check(actual == expected, f"{stat}, expected {expected}, got {actual}")

    print(f"{checks_passed}/{checks_total} checks passed")

mcdavid_known_totals = {
    "goals": 48,
    "assists": 90,
    "points": 138,
    "plusMinus": 17,
    "pim": 44,
    "shots": 306,
    "powerPlayGoals": 13,
    "gameWinningGoals": 4,
    "hits": 40,           # fill in from nhl.com/stats once you check
    "blockedShots": 30,   # same
}

check_against_known_totals(mcdavid_full, mcdavid_known_totals, "McDavid")
print(f"\nGames in our data: {len(mcdavid_full)}, actual games played: 82")

## Section 6.7, testing on random players

Everything so far has been tested on hand-picked players, McDavid, a
specific goalie, Weegar. Worth checking the pipeline holds up on
players we didn't choose, since edge cases tend to hide in the players
nobody thinks to test. This can't check against known real-world
totals, since we don't have those memorized for random players, but it
can confirm the pipeline runs clean and the internal consistency checks
still pass.

In [ ]:
def pick_random_player(position_filter, min_games=10, season=LAST_SEASON, max_attempts=10):
    """
    Picks a random player matching a position filter, skips anyone with
    too few games in the given season, a very short sample isn't a
    useful test of the normal pipeline, that's its own edge case,
    already covered separately with Brossoit earlier.
    """
    candidates = all_players[all_players["position"].isin(position_filter)]

    for _ in range(max_attempts):
        candidate = candidates.sample(1).iloc[0]
        log = get_game_log(candidate["player_id"], season=season, force_refresh=False)
        if len(log) >= min_games:
            print(f"Picked {candidate['name']} ({candidate['position']}), {len(log)} games")
            return candidate["player_id"], candidate["name"]
        print(f"Skipping {candidate['name']}, only {len(log)} games")

    raise RuntimeError(f"Couldn't find a player with at least {min_games} games after {max_attempts} attempts")

random_forward_id, random_forward_name = pick_random_player(["C", "L", "R"])
random_defenseman_id, random_defenseman_name = pick_random_player(["D"])
random_goalie_id, random_goalie_name = pick_random_player(["G"])

In [ ]:
random_forward_log = build_full_skater_log(random_forward_id, LAST_SEASON)
random_forward_log["fantasy_points"] = apply_scoring(random_forward_log, skater_scoring, SKATER_FIELD_MAP)
check_internal_consistency(random_forward_log, random_forward_name)

random_defenseman_log = build_full_skater_log(random_defenseman_id, LAST_SEASON)
random_defenseman_log["fantasy_points"] = apply_scoring(random_defenseman_log, skater_scoring, SKATER_FIELD_MAP)
check_internal_consistency(random_defenseman_log, random_defenseman_name)

goalie_ok = check_game_log_pipeline(
    random_goalie_id, random_goalie_name,
    GOALIE_NUMERIC_COLUMNS, season=LAST_SEASON,
    required_columns=GOALIE_REQUIRED_COLUMNS
)

print(f"\n{random_forward_name}: avg fantasy points {random_forward_log['fantasy_points'].mean():.2f}")
print(f"{random_defenseman_name}: avg fantasy points {random_defenseman_log['fantasy_points'].mean():.2f}")

In [ ]:
random_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=False)
random_goalie_log = clean_goalie_log(random_goalie_log, GOALIE_NUMERIC_COLUMNS)
random_goalie_log["fantasy_points"] = apply_scoring(random_goalie_log, goalie_scoring, GOALIE_FIELD_MAP)

print(f"{random_goalie_name}: {len(random_goalie_log)} games, avg fantasy points: {random_goalie_log['fantasy_points'].mean():.2f}")
random_goalie_log[["gameDate", "decision", "savePctg", "goalsAgainst", "fantasy_points"]].head(4)

In [ ]:
def apply_scoring_breakdown(games_df, scoring_config, field_map):
    """
    Same logic as apply_scoring, but returns a DataFrame with one column
    per scored category showing its point contribution, plus a total
    column, instead of collapsing straight to a single number. Useful
    for checking that every category is actually contributing something,
    not just trusting the final total.
    """
    breakdown = pd.DataFrame(index=games_df.index)
    unmapped = []

    for stat, weight in scoring_config.items():
        field = field_map.get(stat)

        if stat == "W":
            breakdown[f"{stat}_pts"] = (games_df["decision"] == "W").astype(int) * weight
        elif stat == "L":
            breakdown[f"{stat}_pts"] = (games_df["decision"] == "L").astype(int) * weight
        elif stat == "SV":
            breakdown[f"{stat}_pts"] = (games_df["shotsAgainst"] - games_df["goalsAgainst"]) * weight
        elif field is None:
            unmapped.append(stat)
            breakdown[f"{stat}_pts"] = 0.0
        else:
            breakdown[f"{stat}_pts"] = games_df[field] * weight

    breakdown["total"] = breakdown.sum(axis=1)

    if unmapped:
        print(f"Note, these categories aren't available and are showing as 0, not actually scored: {unmapped}")

    return breakdown

In [ ]:
random_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=False)
random_goalie_log = clean_goalie_log(random_goalie_log, GOALIE_NUMERIC_COLUMNS)

scoring_breakdown = apply_scoring_breakdown(random_goalie_log, goalie_scoring, GOALIE_FIELD_MAP)
random_goalie_log["fantasy_points"] = scoring_breakdown["total"]

print(f"{random_goalie_name}: {len(random_goalie_log)} games, avg fantasy points: {random_goalie_log['fantasy_points'].mean():.2f}")

detailed_view = pd.concat([
    random_goalie_log[["gameDate", "decision", "savePctg", "goalsAgainst"]],
    scoring_breakdown
], axis=1)

detailed_view.head(4)

In [ ]:
raw_goalie_log = get_game_log(random_goalie_id, season=LAST_SEASON, force_refresh=True)
missing_rows = raw_goalie_log[raw_goalie_log[GOALIE_REQUIRED_COLUMNS].isna().any(axis=1)]
missing_rows

In [ ]:
def get_player_total_points(player_id, name, position, season=LAST_SEASON):
    if position == "G":
        log = get_game_log(player_id, season=season, force_refresh=False)
        if log.empty:
            return 0, 0.0
        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
    else:
        log = build_full_skater_log(player_id, season)
        if log.empty:
            return 0, 0.0
        log["fantasy_points"] = apply_scoring(log, skater_scoring, SKATER_FIELD_MAP)

    return len(log), log["fantasy_points"].sum()

In [ ]:
random_sample = all_players.sample(5)

print(f"{'Name':<25}{'Pos':<6}{'Team':<6}{'Games':<8}{'Total Points'}")
print("-" * 60)

for _, player in random_sample.iterrows():
    games, total = get_player_total_points(player["player_id"], player["name"], player["position"])
    print(f"{player['name']:<25}{player['position']:<6}{player['team']:<6}{games:<8}{total:.1f}")

Section 6 recap, scoring turned out to need more than a simple lookup
table. A few categories, PPA and SHA specifically, were derivable from
existing fields without any extra API calls. Hits and blocked shots
needed a separate pull entirely, from the boxscore endpoint rather than
the game log. Faceoffs won and lost turned out to not be available as
raw counts anywhere in this API, only as a percentage, honestly flagged
as a known gap rather than approximated. Relief appearances by goalies
needed special handling too, missing a decision doesn't mean bad data,
it means no decision was awarded, and the row needed to be kept, not
dropped. Verified against a real player's known season totals, and
against 5 randomly selected players checked manually against Yahoo,
all matching.

## Section 7, rolling expected value and volatility

This is the mathematical core of the whole model. For every player, two
numbers get tracked, not one, $\mu$, the expected fantasy points per
game, and $\sigma$, how much that varies game to game. Both matter, two
players can average the same points per game and be completely
different assets if one is consistent and the other swings wildly.

### Why not a simple average

A flat average over the last $N$ games treats every game as equally
important. That's not quite right, hockey performance is streaky, a
role change, an injury recovery, a hot line, all make recent games more
informative than a game from a month ago. So instead of a flat average,
this uses an exponentially weighted moving average, EWMA, more recent
games count more, older games count less, and the weight decays
smoothly rather than falling off a cliff at some arbitrary cutoff.

### The EWMA formula

For a sequence of fantasy point totals $x_1, x_2, \dots, x_t$, in
chronological order, the EWMA at time $t$ is defined recursively,

$$
\mu_t = \alpha x_t + (1 - \alpha)\mu_{t-1}
$$

where $\alpha \in (0, 1]$ controls how much weight the newest
observation gets. A higher $\alpha$ means the average reacts faster to
recent games, a lower $\alpha$ means it's smoother and more stable, but
slower to pick up on a real change in form.

Unrolled, this is equivalent to a weighted average of every game so
far, with weights that shrink geometrically going back in time,

$$
\mu_t = \alpha \sum_{i=0}^{t-1} (1 - \alpha)^i \, x_{t-i}
$$

normalized so the weights sum to 1. That normalization matters early
on, with only a handful of games, without it the estimate would be
biased toward zero, pandas handles this correction automatically, more
on that below.

### Choosing alpha via a half-life

$\alpha$ isn't very intuitive on its own, a more natural way to set it
is by choosing a half-life, $h$, the number of games after which a
game's weight has decayed to half its original influence. The
relationship is,

$$
\alpha = 1 - \left(\frac{1}{2}\right)^{1/h}
$$

For example, a half-life of 5 games means a game from 5 games ago
counts half as much as the most recent game, and a game from 10 games
ago counts a quarter as much. This is a much easier thing to reason
about and explain than a raw alpha value, and it's the number that'll
actually get tuned later if the model needs adjusting.

### Volatility, sigma, under the same weighting

Sigma uses the same exponential weighting, not a flat standard
deviation, for the same reason mu does, a player's recent volatility is
more relevant than volatility from two months ago. The exponentially
weighted variance follows a similar recursive form,

$$
\sigma_t^2 = \alpha (x_t - \mu_{t-1})^2 + (1 - \alpha)\sigma_{t-1}^2
$$

and $\sigma_t = \sqrt{\sigma_t^2}$. Pandas computes both $\mu$ and
$\sigma$ this way directly, using `.ewm()`, so this doesn't need to be
hand-rolled, just understood.

In [ ]:
def compute_ewma_mu_sigma(points_series, half_life=5):
    """
    Computes the exponentially weighted mean (mu) and standard deviation
    (sigma) of a fantasy points series, most recent games weighted most
    heavily. half_life is in games, not a raw alpha, since half-life is
    the more interpretable knob to tune.
    """
    ewm = points_series.ewm(halflife=half_life, adjust=True)
    mu = ewm.mean()
    sigma = ewm.std()
    return mu, sigma

# test on McDavid's already-built season
mcdavid_mu, mcdavid_sigma = compute_ewma_mu_sigma(mcdavid_full["fantasy_points"])

mcdavid_full["mu"] = mcdavid_mu
mcdavid_full["sigma"] = mcdavid_sigma

mcdavid_full[["gameDate", "fantasy_points", "mu", "sigma"]].tail(10)

In [ ]:
mcdavid_full[["gameDate", "fantasy_points", "mu", "sigma"]].head(10)

In [ ]:
raw_ewm = mcdavid_full["fantasy_points"].ewm(halflife=5, adjust=False).mean()
corrected_ewm = mcdavid_full["fantasy_points"].ewm(halflife=5, adjust=True).mean()

pd.DataFrame({
    "fantasy_points": mcdavid_full["fantasy_points"],
    "raw (adjust=False)": raw_ewm,
    "corrected (adjust=True)": corrected_ewm
}).head(10)

In [ ]:
comparison = pd.DataFrame({
    "fantasy_points": mcdavid_full["fantasy_points"],
    "raw (adjust=False)": raw_ewm,
    "corrected (adjust=True)": corrected_ewm,
    "gap": corrected_ewm - raw_ewm
})
comparison.tail(15)

A quick side by side check confirms this, adjust=False, the plain
recursive formula, and adjust=True, pandas' early-sample corrected
version, diverge noticeably in the first several games, at game 1 they
differ by about 0.02, but by roughly game 20 to 30 the gap has shrunk
to a fraction of a point, and by game 65 to 70 onward it's on the order
of 1e-5, effectively zero. The correction matters early, exactly when
there isn't much history to work with yet, and becomes irrelevant once
enough games have accumulated.

In [ ]:
mcdavid_full[mcdavid_full["fantasy_points"] < 0][["gameDate", "goals", "assists", "pim", "fantasy_points"]]

## Section 8, shrinkage toward a prior season

Early in a season, there aren't enough games to trust mu and sigma on
their own, game one alone gives no sigma at all, as seen already.
Shrinkage solves this by blending in last season's stats as a prior,
leaned on heavily early, fading out as real current-season games
accumulate.

### The blending formula

$$
\hat{\mu} = \frac{n \cdot \mu_{\text{current}} + k \cdot \mu_{\text{prior}}}{n + k}
$$

where $n$ is the number of current-season games observed so far, and
$k$ is a constant controlling how strongly the prior is trusted. A
larger $k$ means the prior dominates longer before current-season data
takes over, a smaller $k$ means current data takes over faster. The
same blend applies to the variance,

$$
\hat{\sigma}^2 = \frac{n \cdot \sigma^2_{\text{current}} + k \cdot \sigma^2_{\text{prior}}}{n + k}
$$

and $\hat{\sigma} = \sqrt{\hat{\sigma}^2}$.

Two limiting cases worth checking by hand, when $n = 0$, no current
season games yet, $\hat{\mu} = \mu_{\text{prior}}$ exactly, the
estimate is entirely the prior, which is correct, there's nothing else
to go on. As $n \to \infty$, the $k \cdot \mu_{\text{prior}}$ term
becomes negligible next to $n \cdot \mu_{\text{current}}$, and
$\hat{\mu} \to \mu_{\text{current}}$, the prior fades out entirely,
also correct.

### Testing this without a real current season

The actual current season, 2026-27, has zero games right now, so there
isn't real partial-season data to test blending against yet. To confirm
the mechanism actually works before the real season gives us genuine
partial data, this simulates it, treating 2024-25 as the prior and
2025-26 as if it were still in progress, checking the blend at a few
different points partway through that season.

In [ ]:
PRIOR_SEASON = "20242025"

def get_prior_season_stats(player_id, season, position):
    """
    Computes simple season-long mean and std from a completed season,
    used as the shrinkage prior. Not EWMA, the season is already over,
    so there's no recency to weight, just the overall level and spread.
    """
    if position == "G":
        log = get_game_log(player_id, season=season, force_refresh=False)
        if log.empty:
            return None, None
        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
    else:
        log = build_full_skater_log(player_id, season)
        if log.empty:
            return None, None
        log["fantasy_points"] = apply_scoring(log, skater_scoring, SKATER_FIELD_MAP)

    return log["fantasy_points"].mean(), log["fantasy_points"].std()

def blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n, k=10):
    blended_mu = (n * current_mu + k * prior_mu) / (n + k)
    blended_var = (n * current_sigma**2 + k * prior_sigma**2) / (n + k)
    return blended_mu, blended_var**0.5



In [ ]:
prior_mu, prior_sigma = get_prior_season_stats(skater_id, PRIOR_SEASON, "C")
print(f"McDavid, 2024-25 prior, mu: {prior_mu:.2f}, sigma: {prior_sigma:.2f}")

for n_games in [1, 5, 10, 20, 40, 82]:
    partial_log = mcdavid_full.iloc[:n_games]
    current_mu = partial_log["fantasy_points"].mean()
    current_sigma = partial_log["fantasy_points"].std() if n_games > 1 else 0

    blended_mu, blended_sigma = blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n=n_games, k=10)

    print(f"n={n_games:>3}, current mu={current_mu:.2f}, blended mu={blended_mu:.2f}, "
          f"current sigma={current_sigma:.2f}, blended sigma={blended_sigma:.2f}")

In [ ]:
print((82 * 2.75 + 10 * 2.23) / 92)

## Visualizing k, how fast should the prior fade

Rather than picking k by intuition, worth actually seeing how different
values behave. A small k lets current-season data take over almost
immediately, a large k keeps leaning on last season for a long time.
Plotting a few options side by side against the actual game log makes
this a visible, informed choice, not a guess.

In [ ]:
import matplotlib.pyplot as plt

k_values = [2, 5, 10, 20, 40]
games_range = range(1, 83)

plt.figure(figsize=(10, 6))

# reference lines, the two limiting cases the formula should approach
plt.axhline(prior_mu, color="gray", linestyle="--", label="prior mu (2024-25)")
final_season_mu = mcdavid_full["fantasy_points"].mean()
plt.axhline(final_season_mu, color="black", linestyle=":", label="final current-season mu")

for k in k_values:
    blended_values = []
    for n_games in games_range:
        partial_log = mcdavid_full.iloc[:n_games]
        current_mu = partial_log["fantasy_points"].mean()
        current_sigma = partial_log["fantasy_points"].std() if n_games > 1 else 0
        blended_mu, _ = blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n=n_games, k=k)
        blended_values.append(blended_mu)
    plt.plot(games_range, blended_values, label=f"k = {k}")

plt.xlabel("Games played this season")
plt.ylabel("Blended mu")
plt.title("How different k values let go of the prior over the season")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## A tempting alternative, blending with EWMA instead

The natural next thought is to blend the prior with the EWMA mu from
Section 7 instead of the flat cumulative mean, one number that's both
recency-weighted and prior-informed. Tried this, and it didn't hold up,
details below.

## Correction, EWMA and shrinkage answer different questions

An earlier version of this notebook blended the prior with the EWMA
value, expecting it to be smoother than blending with a flat cumulative
mean. Plotting it showed the opposite, blending with EWMA stayed
volatile all season, never settling down. The reason, EWMA's alpha
stays constant regardless of how many games have been played, so it
never stops reacting to recent streaks, while a flat mean's sensitivity
to any single new game shrinks as 1/n, naturally stabilizing over the
season.

That means these two are actually answering different questions, EWMA
answers, how is this player doing right now, recently. A cumulative
mean answers, what is this player's underlying level this season,
overall. Shrinkage is a season-level question, so it belongs with the
flat mean, not EWMA. Going forward, both get computed and kept
separate, blended mu, for season-level talent estimates and
comparisons, and EWMA mu, for a recent hot or cold signal, used
differently, not combined into one number.

## Choosing k = 10

Comparing k values on real season data, k=2 overreacts to hot and cold
stretches that are ultimately just noise, spiking well above the
eventual season average around games 35-45. k=40 stays the most stable
but is noticeably slow to trust a real, sustained improvement, still
sitting below the final average even after a full 82 games. k=10 sits
in between, damping early noise while still converging reasonably close
to the final season mu by the end of the year.

This is a judgment call, not something derived mathematically, worth
revisiting once real weekly predictions accumulate this season, if
predictions consistently lag behind what's actually happening, that's
a sign k is too large, if they overreact to one good or bad week, k is
too small. Logged here as v1's choice, k=10, open to tuning based on
actual calibration once there's real data to check it against.

In [ ]:
def get_player_estimates(player_id, position, season=LAST_SEASON, prior_season=PRIOR_SEASON, half_life=5, k=10):
    """
    Computes both estimates for a player, EWMA (recent form) and
    shrinkage-blended (season-level talent, leaning on last season
    early). This is the actual final output of everything built in
    Sections 5 through 8, what gets fed into player comparisons next.
    """
    if position == "G":
        log = get_game_log(player_id, season=season, force_refresh=False)
        if log.empty:
            return None
        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
    else:
        log = build_full_skater_log(player_id, season)
        if log.empty:
            return None
        log["fantasy_points"] = apply_scoring(log, skater_scoring, SKATER_FIELD_MAP)

    prior_mu, prior_sigma = get_prior_season_stats(player_id, prior_season, position)

    ewma_mu, ewma_sigma = compute_ewma_mu_sigma(log["fantasy_points"], half_life=half_life)

    n_games = len(log)
    current_mu = log["fantasy_points"].mean()
    current_sigma = log["fantasy_points"].std() if n_games > 1 else 0

    if prior_mu is not None:
        blended_mu, blended_sigma = blend_mu_sigma(current_mu, current_sigma, prior_mu, prior_sigma, n=n_games, k=k)
    else:
        # no prior season available, fall back to current-season-only estimate
        blended_mu, blended_sigma = current_mu, current_sigma

    return {
        "games_played": n_games,
        "recent_mu": ewma_mu.iloc[-1],
        "recent_sigma": ewma_sigma.iloc[-1],
        "season_mu": blended_mu,
        "season_sigma": blended_sigma,
    }

In [ ]:
skater_id = lookup_player_id("McDavid")
weegar_id = lookup_player_id("Weegar")
weegar_name = all_players[all_players["player_id"] == weegar_id].iloc[0]["name"]

# make sure we have a goalie id/name to test with, in case these got lost earlier
goalie_id = all_players[all_players["position"] == "G"].iloc[0]["player_id"]
goalie_name = all_players[all_players["position"] == "G"].iloc[0]["name"]

mcdavid_estimates = get_player_estimates(skater_id, "C")
weegar_estimates = get_player_estimates(weegar_id, "D")
goalie_estimates = get_player_estimates(goalie_id, "G")

print("McDavid:", mcdavid_estimates)
print(f"{weegar_name}:", weegar_estimates)
print(f"{goalie_name}:", goalie_estimates)

## Section 8 recap, two numbers, two different jobs

At this point there are two separate ways of estimating a player's
performance, and it's worth being explicit that both are being kept,
on purpose, rather than picking one.

EWMA mu and sigma, from Section 7, answer, how is this player doing
right now, recently, reacting quickly to recent form.

Blended mu and sigma, from this section, answer, what is this player's
underlying level this season overall, leaning on last season early,
shifting to this season's own data as it accumulates, deliberately
more stable and slower to react than EWMA.

Both get computed for every player going forward, not combined into
one number, since they answer different questions and both matter for
different kinds of calls.

## Section 9, comparing two players, a real probability, not just a bigger number

Given two players' mu and sigma, the question isn't just who has the
higher expected value, it's how confident should you actually be. Two
players with a big mu gap but also big sigma could be a coin flip in
practice, two players with a small $\mu$ gap but small sigma could be a
near-certainty.

$$
P(A > B) = \Phi\left( \frac{\mu_A - \mu_B}{\sqrt{\sigma_A^2 + \sigma_B^2}} \right)
$$

where $\Phi$ is the standard normal cumulative distribution function.
This treats the difference between two players' game-to-game output as
approximately normally distributed, an approximation, not an exact
model of reality, already flagged as a known simplification in
METHODOLOGY.md.

Worth deciding which mu/sigma feeds this, season-level (blended) or
recent (EWMA). Season-level is the default here, it's the more stable,
better-supported estimate, meant to answer "who is the better fantasy
asset right now, overall." Recent form is worth surfacing alongside it
as extra context, useful for a human making the final call, rather
than swapped in as the primary number the probability is based on.

In [ ]:
from scipy.stats import norm

def compare_players(estimates_a, estimates_b, label_a="Player A", label_b="Player B", use="season"):
    mu_key = f"{use}_mu"
    sigma_key = f"{use}_sigma"

    mu_a, sigma_a = estimates_a[mu_key], estimates_a[sigma_key]
    mu_b, sigma_b = estimates_b[mu_key], estimates_b[sigma_key]

    combined_sigma = (sigma_a**2 + sigma_b**2) ** 0.5

    if combined_sigma == 0:
        # both perfectly consistent, no uncertainty to model, whoever has
        # the higher mu wins outright, a tie is a genuine coin flip
        probability_a_wins = 1.0 if mu_a > mu_b else (0.0 if mu_a < mu_b else 0.5)
    else:
        z = (mu_a - mu_b) / combined_sigma
        probability_a_wins = norm.cdf(z)

    return {
        "label_a": label_a, "mu_a": mu_a, "sigma_a": sigma_a,
        "label_b": label_b, "mu_b": mu_b, "sigma_b": sigma_b,
        "probability_a_outperforms_b": probability_a_wins,
    }

def print_comparison(result):
    print(f"{result['label_a']}: mu={result['mu_a']:.2f}, sigma={result['sigma_a']:.2f}")
    print(f"{result['label_b']}: mu={result['mu_b']:.2f}, sigma={result['sigma_b']:.2f}")
    print(f"P({result['label_a']} outperforms {result['label_b']}) = {result['probability_a_outperforms_b']:.1%}")

In [ ]:
# a clear case, should come back with high confidence
result = compare_players(mcdavid_estimates, weegar_estimates, "McDavid", weegar_name)
print_comparison(result)

print()

# a closer case, worth finding two players nearer in mu to see a less extreme probability
skater_candidates = all_players[~all_players["position"].isin(["G"])].sample(2)
player_x_id, player_x_name = skater_candidates.iloc[0]["player_id"], skater_candidates.iloc[0]["name"]
player_y_id, player_y_name = skater_candidates.iloc[1]["player_id"], skater_candidates.iloc[1]["name"]

player_x_position = skater_candidates.iloc[0]["position"]
player_y_position = skater_candidates.iloc[1]["position"]

estimates_x = get_player_estimates(player_x_id, player_x_position)
estimates_y = get_player_estimates(player_y_id, player_y_position)

if estimates_x and estimates_y:
    result2 = compare_players(estimates_x, estimates_y, player_x_name, player_y_name)
    print_comparison(result2)
else:
    print("One of these players has no games this season, try running this cell again for a different pair")

In [ ]:
#Should match above
import math
z = (result["mu_a"] - result["mu_b"]) / math.sqrt(result["sigma_a"]**2 + result["sigma_b"]**2)
print(norm.cdf(z))

## Section 9.5, testing the comparison model rigorously

Two kinds of checks here. First, mathematical properties the formula
must satisfy no matter what data goes in, symmetry, a fair comparison
between identical players, more uncertainty pulling confidence toward
a coin flip. These aren't about any specific player, they're checking
the function itself is correct.

Second, a sanity check on the actual number just produced. 69.9% for
McDavid over Weegar feels lower than intuition suggests for a gap this
large, worth understanding why before trusting it, not dismissing it
because it seems low.

In [ ]:
def check_comparison_properties():
    checks_passed = 0
    checks_total = 0

    def check(condition, description):
        nonlocal checks_passed, checks_total
        checks_total += 1
        if condition:
            checks_passed += 1
            print(f"[pass] {description}")
        else:
            print(f"[FAIL] {description}")

    # symmetry, P(A>B) and P(B>A) must sum to 1
    a = {"season_mu": 3.0, "season_sigma": 1.5}
    b = {"season_mu": 2.0, "season_sigma": 1.0}
    result_ab = compare_players(a, b)
    result_ba = compare_players(b, a)
    check(abs(result_ab["probability_a_outperforms_b"] + result_ba["probability_a_outperforms_b"] - 1.0) < 1e-9,
          "P(A>B) and P(B>A) sum to exactly 1")

    # identical players, must be exactly 50%
    same = {"season_mu": 2.5, "season_sigma": 1.2}
    result_same = compare_players(same, dict(same))
    check(abs(result_same["probability_a_outperforms_b"] - 0.5) < 1e-9,
          "two identical players give exactly 50%")

    # higher mu, same sigma, must push probability above 50%
    higher_mu = {"season_mu": 4.0, "season_sigma": 1.2}
    result_higher = compare_players(higher_mu, same)
    check(result_higher["probability_a_outperforms_b"] > 0.5,
          "a strictly higher mu, same sigma, gives above 50%")

    # increasing sigma, same mu gap, should pull the probability toward 50%
    low_sigma_a = {"season_mu": 3.0, "season_sigma": 0.5}
    low_sigma_b = {"season_mu": 2.0, "season_sigma": 0.5}
    high_sigma_a = {"season_mu": 3.0, "season_sigma": 3.0}
    high_sigma_b = {"season_mu": 2.0, "season_sigma": 3.0}

    result_low_sigma = compare_players(low_sigma_a, low_sigma_b)
    result_high_sigma = compare_players(high_sigma_a, high_sigma_b)
    check(result_low_sigma["probability_a_outperforms_b"] > result_high_sigma["probability_a_outperforms_b"],
          "more volatility, same mu gap, pulls confidence closer to 50%")

    # zero volatility on both sides, should be a near-certain call, not undefined
    try:
        deterministic_a = {"season_mu": 3.0, "season_sigma": 0.0}
        deterministic_b = {"season_mu": 2.0, "season_sigma": 0.0}
        result_det = compare_players(deterministic_a, deterministic_b)
        check(result_det["probability_a_outperforms_b"] > 0.99,
              "zero volatility on both sides doesn't crash, and gives near-certainty")
    except ZeroDivisionError:
        check(False, "zero volatility on both sides doesn't crash, and gives near-certainty")

    print(f"\n{checks_passed}/{checks_total} checks passed")

check_comparison_properties()

## Testing player comparisons on random players

Same idea as the random player tests back in Section 6, pick players
we didn't choose and see if the comparison holds up and produces
sensible, explainable results, not just numbers that don't crash.

In [ ]:
def get_random_comparable_pair(position_filter, min_games=10, max_attempts=15):
    """
    Picks two random players of the same position group with at least
    min_games this season, so the comparison isn't distorted by a
    tiny, unstable sample on either side.
    """
    candidates = all_players[all_players["position"].isin(position_filter)]

    valid_players = []
    attempts = 0
    while len(valid_players) < 2 and attempts < max_attempts:
        candidate = candidates.sample(1).iloc[0]
        estimates = get_player_estimates(candidate["player_id"], candidate["position"])
        if estimates and estimates["games_played"] >= min_games:
            valid_players.append((candidate["player_id"], candidate["name"], candidate["position"], estimates))
        attempts += 1

    if len(valid_players) < 2:
        return None

    return valid_players[0], valid_players[1]

for i in range(5):
    pair = get_random_comparable_pair(["C", "L", "R", "D"])
    if pair is None:
        print(f"Pair {i+1}: couldn't find two valid players, skipping")
        continue

    (id_a, name_a, pos_a, est_a), (id_b, name_b, pos_b, est_b) = pair
    result = compare_players(est_a, est_b, name_a, name_b)
    print(f"\n--- Pair {i+1} ---")
    print_comparison(result)

In [ ]:
schmidt_id = 8477220
schenn_id = 8474568
schmidt_log = build_full_skater_log(schmidt_id, LAST_SEASON)  # use your actual variable names here
schenn_log = build_full_skater_log(schenn_id, LAST_SEASON)

schmidt_breakdown = apply_scoring_breakdown(schmidt_log, skater_scoring, SKATER_FIELD_MAP)
schenn_breakdown = apply_scoring_breakdown(schenn_log, skater_scoring, SKATER_FIELD_MAP)

comparison = pd.DataFrame({
    "Schmidt total": schmidt_breakdown.sum(),
    "Schenn total": schenn_breakdown.sum(),
})
comparison["Schmidt per game"] = comparison["Schmidt total"] / len(schmidt_log)
comparison["Schenn per game"] = comparison["Schenn total"] / len(schenn_log)

schmidt_log["fantasy_points"] = schmidt_breakdown["total"]
schenn_log["fantasy_points"] = schenn_breakdown["total"]

comparison


In [ ]:
schmidt_prior_mu, schmidt_prior_sigma = get_prior_season_stats(schmidt_id, PRIOR_SEASON, "D")
schenn_prior_mu, schenn_prior_sigma = get_prior_season_stats(schenn_id, PRIOR_SEASON, "D")

print(f"Schmidt, current season games: {len(schmidt_log)}, current mu: {schmidt_log['fantasy_points'].mean():.2f}, prior mu: {schmidt_prior_mu}")
print(f"Schenn, current season games: {len(schenn_log)}, current mu: {schenn_log['fantasy_points'].mean():.2f}, prior mu: {schenn_prior_mu}")

## A real example, peripherals closing a production gap

Comparing Nate Schmidt and Luke Schenn, two defensemen with very
different offensive production (Schmidt, 10 goals and 17 assists this
season, versus Schenn's 2 goals and 6 assists), the model still rates
them nearly identically, 0.84 vs 0.83 fantasy points per game.

The breakdown shows why, Schenn earns roughly three times as many
points per game from hits alone as Schmidt does, closing almost the
entire gap Schmidt built through actual scoring. A secondary factor,
Schenn's prior season was notably stronger than his current one, while
Schmidt is outperforming his own prior, nudging the shrinkage estimate
slightly further in Schenn's favor.

This isn't a bug, it's the scoring config doing exactly what it was
told to value. Worth surfacing as a real, explainable finding, not
something to quietly override, this league's scoring genuinely rewards
physical defensemen close to offensive ones, and the model correctly
reflects that.

## Section 9 recap, a real comparison model, and a real example of it working

Built P(A > B) using the normal approximation from METHODOLOGY.md,
verified against five formal properties, symmetry, identical players
giving exactly 50%, higher mu pushing above 50%, more volatility
pulling toward 50%, and zero volatility not crashing, a real bug that
turned up and got fixed, division by zero when comparing two perfectly
consistent players.

Testing on random pairs surfaced a genuinely useful case, Nate Schmidt
and Luke Schenn came back nearly identical, 0.84 vs 0.83 fantasy points
per game, despite very different offensive production. Traced this to
two real causes, hits contributing roughly three times more to
Schenn's total than Schmidt's under this league's scoring, closing most
of the gap Schmidt built through actual goals and assists, and a
smaller effect from the shrinkage prior, since Schenn's prior season
was stronger than his current one while Schmidt is outperforming his
own prior.

Worth being clear about what the probability from compare_players
actually claims, P(A outperforms B) in a single game, not "who's the
better player." A large talent gap can still produce a modest
single-game probability, since per-game variance is large for
everyone in this sport, this is correct behavior, not a flaw, and
worth remembering whenever a number looks lower than intuition expects.

With this, the pipeline built in Sections 5 through 9 is genuinely end
to end, pull real data, clean it, score it under any league's rules,
estimate expected performance and volatility two different ways, and
compare any two players with a real, tested probability. Next, Section
10 adds the two goalie adjustments already promised in METHODOLOGY.md
but not yet built, rest and opponent strength.

## Section 10, goalie adjustments, rest and opponent strength

Two adjustments promised in METHODOLOGY.md but not yet built. Both are
meant to sharpen a goalie's expected performance for a specific
upcoming start, not change the season-long mu/sigma from Section 8.

Back-to-back starts don't need any new data, a goalie's own game log
already has the date of every game, a rest gap of one day or less
between consecutive appearances is a back-to-back. Rather than assume
a fixed penalty, the actual size of the effect gets measured directly
from the pulled data, comparing average fantasy points on back-to-backs
against everything else.

Opponent strength needs a new pull, team-level goals scored per game,
league-wide, from the NHL's standings endpoint. A goalie facing a team
that scores a lot should have a lower expectation than one facing a
weak offense, this adjusts expected goals against accordingly.

In [ ]:
def flag_back_to_backs(log_df):
    """
    Flags each game as a back-to-back if it falls one day or less after
    the player's previous game. Needs the log sorted by date, which
    clean_goalie_log already guarantees.
    """
    df = log_df.copy()
    rest_days = df["gameDate"].diff().dt.days
    df["is_back_to_back"] = rest_days <= 1
    df.loc[df.index[0], "is_back_to_back"] = False  # first game has no prior game to compare to
    return df

In [ ]:
# Note, some goalies (especially backups with few starts) will have zero
# back-to-backs of their own. That's fine here, since we're pooling every
# individual game across all goalies into one list, not averaging each
# goalie's own back-to-back rate. A goalie contributing 0 back-to-back
# games just contributes 0 games to that list, same as Brossoit above.
def measure_back_to_back_effect(log_df):
    """
    Compares average fantasy points on back-to-backs vs everything else,
    for one goalie. This is the empirical estimate mentioned in
    METHODOLOGY.md, not an assumed number.
    """
    b2b_avg = log_df[log_df["is_back_to_back"]]["fantasy_points"].mean()
    rested_avg = log_df[~log_df["is_back_to_back"]]["fantasy_points"].mean()
    b2b_count = log_df["is_back_to_back"].sum()

    print(f"Back-to-back starts: {b2b_count} games, avg fantasy points: {b2b_avg:.2f}")
    print(f"All other starts: {len(log_df) - b2b_count} games, avg fantasy points: {rested_avg:.2f}")
    print(f"Difference: {b2b_avg - rested_avg:.2f}")

    return b2b_avg, rested_avg

# test on the goalie already loaded from Section 6
goalie_log = get_game_log(goalie_id, season=LAST_SEASON, force_refresh=False)
goalie_log = clean_goalie_log(goalie_log, GOALIE_NUMERIC_COLUMNS)
goalie_log["fantasy_points"] = apply_scoring(goalie_log, goalie_scoring, GOALIE_FIELD_MAP)
goalie_log = flag_back_to_backs(goalie_log)

measure_back_to_back_effect(goalie_log)

In [ ]:
def pool_back_to_back_effect(season=LAST_SEASON, min_games=15, pause=0.5):
    all_goalies = all_players[all_players["position"] == "G"]

    b2b_points = []
    rested_points = []

    for _, goalie in all_goalies.iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
        log = flag_back_to_backs(log)

        b2b_points.extend(log[log["is_back_to_back"]]["fantasy_points"].tolist())
        rested_points.extend(log[~log["is_back_to_back"]]["fantasy_points"].tolist())

    b2b_avg = sum(b2b_points) / len(b2b_points)
    rested_avg = sum(rested_points) / len(rested_points)

    print(f"Pooled across {len(all_goalies)} goalies")
    print(f"Back-to-back starts: {len(b2b_points)} games, avg fantasy points: {b2b_avg:.2f}")
    print(f"Rested starts: {len(rested_points)} games, avg fantasy points: {rested_avg:.2f}")
    print(f"League-wide effect: {b2b_avg - rested_avg:.2f} fantasy points")

    return b2b_avg - rested_avg, b2b_points, rested_points

In [ ]:
from scipy import stats
back_to_back_penalty, b2b_points, rested_points = pool_back_to_back_effect()
t_stat, p_value = stats.ttest_ind(b2b_points, rested_points, equal_var=False)
print(f"t-statistic: {t_stat:.2f}, p-value: {p_value:.3f}")

In [ ]:
# is the same set of goalies well-represented in both groups, or is the
# back-to-back bucket dominated by a different population of goalies?
b2b_goalie_ids = set()
for _, goalie in all_players[all_players["position"] == "G"].iterrows():
    log = get_game_log(goalie["player_id"], season=LAST_SEASON, force_refresh=False)
    if len(log) < 15:
        continue
    log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
    log = flag_back_to_backs(log)
    if log["is_back_to_back"].sum() > 0:
        b2b_goalie_ids.add(goalie["player_id"])

print(f"{len(b2b_goalie_ids)} distinct goalies contributed at least one back-to-back game")

# Back-to-Back Goalie Adjustment

Status: tested, not implemented, insufficient evidence

## What was tried
Pooled fantasy points across all goalies with 15+ games last season,
comparing back-to-back starts (53 games) against all other starts
(2,521 games). Observed a -0.34 point difference, but a two-sample
t-test came back with p = 0.560, statistically indistinguishable from
no effect at all given this sample size.

## Why this isn't built into v1
53 back-to-back games league-wide in a season isn't enough to detect
an effect of this likely size, if a real effect exists, even a
meaningful one, distinguishing it from noise would need either more
seasons of pooled data, or a much larger sample than one season alone
provides.

## Revisit when
Once multiple seasons of pooled back-to-back data are available, or if
the weekly prediction review process surfaces goalies on back-to-backs
consistently underperforming their season-level mu, a real pattern
worth then testing formally rather than assumed from a small sample.

## Opponent strength, goalies first

Every game a goalie plays has an opponent, unlike back-to-backs, this
isn't limited to a small subset of games, so there's a much larger
sample to test against. The idea, a goalie facing a high-scoring
offense should have a higher expected goals-against than one facing a
weak offense, this pulls in team-level scoring data to test that
directly, rather than assuming it.

In [ ]:
def get_team_offense_stats(season_end_date):
    """
    Pulls season-to-date team stats from the standings endpoint, goals
    for and against per game, per team. Used as the opponent-strength
    signal for both goalies (opponent's goals-for) and, later, skaters
    (opponent's goals-against).
    """
    url = f"https://api-web.nhle.com/v1/standings/{season_end_date}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    rows = []
    for team in data.get("standings", []):
        games_played = team.get("gamesPlayed", 0)
        if games_played == 0:
            continue
        rows.append({
            "team": team["teamAbbrev"]["default"],
            "goals_for_per_game": team["goalFor"] / games_played,
            "goals_against_per_game": team["goalAgainst"] / games_played,
        })

    return pd.DataFrame(rows)

team_offense = get_team_offense_stats("2026-04-17")
print(len(team_offense))
team_offense.head()

## Testing whether opponent offense actually predicts goalie performance

Same standard as the back-to-back check, don't assume the relationship
exists just because it's intuitive, test it directly. If a goalie's
fantasy points per game correlate with how much their opponent
typically scores, that's real evidence worth building on. If not,
this gets shelved the same way back-to-backs did.

In [ ]:
def add_opponent_strength(log_df, team_offense_df):
    """
    Merges the opponent's goals-for-per-game onto each game in a
    goalie's log, using opponentAbbrev to match teams.
    """
    return log_df.merge(
        team_offense_df[["team", "goals_for_per_game"]],
        left_on="opponentAbbrev", right_on="team", how="left"
    ).drop(columns="team")

goalie_log = get_game_log(goalie_id, season=LAST_SEASON, force_refresh=False)
goalie_log = clean_goalie_log(goalie_log, GOALIE_NUMERIC_COLUMNS)
goalie_log["fantasy_points"] = apply_scoring(goalie_log, goalie_scoring, GOALIE_FIELD_MAP)
goalie_log = add_opponent_strength(goalie_log, team_offense)

goalie_log[["gameDate", "opponentAbbrev", "goals_for_per_game", "fantasy_points"]].head()

In [ ]:
def check_opponent_strength_correlation(season=LAST_SEASON, min_games=15, pause=0.5):
    """
    Pools fantasy points and opponent goals-for-per-game across every
    goalie with a reasonable sample, checks whether they're actually
    correlated, rather than assuming a stronger opponent hurts
    performance.
    """
    all_points = []
    all_opponent_strength = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)
        log = add_opponent_strength(log, team_offense)

        valid = log.dropna(subset=["goals_for_per_game"])
        all_points.extend(valid["fantasy_points"].tolist())
        all_opponent_strength.extend(valid["goals_for_per_game"].tolist())

    correlation, p_value = stats.pearsonr(all_opponent_strength, all_points)
    print(f"Pooled across {len(all_points)} goalie starts")
    print(f"Correlation between opponent goals-for and goalie fantasy points: {correlation:.3f}")
    print(f"p-value: {p_value:.4f}")

    return all_points, all_opponent_strength

points, opponent_strength = check_opponent_strength_correlation()

In [ ]:
def check_shots_against_correlation(season=LAST_SEASON, min_games=15, pause=0.5):
    """
    Pools shots against and fantasy points across every goalie with a
    reasonable sample. Note this overlaps mechanically with the SV
    scoring category, more shots faced means more save opportunities
    under this league's scoring, worth keeping in mind when
    interpreting whatever correlation comes out.
    """
    all_points = []
    all_shots_against = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        log["fantasy_points"] = apply_scoring(log, goalie_scoring, GOALIE_FIELD_MAP)

        all_points.extend(log["fantasy_points"].tolist())
        all_shots_against.extend(log["shotsAgainst"].tolist())

    correlation, p_value = stats.pearsonr(all_shots_against, all_points)
    print(f"Pooled across {len(all_points)} goalie starts")
    print(f"Correlation between shots against and fantasy points: {correlation:.3f}")
    print(f"p-value: {p_value:.6f}")

    return all_points, all_shots_against

points_shots, shots_against = check_shots_against_correlation()

In [ ]:
sv_weight = goalie_scoring.get("SV", 0)

def check_sv_component_correlation(season=LAST_SEASON, min_games=15, pause=0.5):
    all_sv_points = []
    all_shots_against = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        sv_points = (log["shotsAgainst"] - log["goalsAgainst"]) * sv_weight

        all_sv_points.extend(sv_points.tolist())
        all_shots_against.extend(log["shotsAgainst"].tolist())

    correlation, p_value = stats.pearsonr(all_shots_against, all_sv_points)
    print(f"Correlation between shots against and SV points alone: {correlation:.3f}, p={p_value:.6f}")

check_sv_component_correlation()

In [ ]:
def check_shots_vs_save_pctg(season=LAST_SEASON, min_games=15, pause=0.5):
    """
    Checks whether shots against correlates with save percentage,
    the actual skill/rhythm question, separate from the mechanical
    SV-points relationship. Drops any game with a missing save
    percentage (e.g., a very short appearance with too few shots for
    the API to compute one) before correlating, rather than letting a
    single NaN silently break the whole calculation.
    """
    all_save_pctg = []
    all_shots_against = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        all_save_pctg.extend(log["savePctg"].tolist())
        all_shots_against.extend(log["shotsAgainst"].tolist())

    valid_pairs = [(s, sv) for s, sv in zip(all_shots_against, all_save_pctg) if not pd.isna(sv)]
    shots_clean, save_pctg_clean = zip(*valid_pairs)
    dropped = len(all_shots_against) - len(shots_clean)

    correlation, p_value = stats.pearsonr(shots_clean, save_pctg_clean)

    print(f"Pooled across {len(shots_clean)} goalie starts ({dropped} dropped for missing save percentage)")
    print(f"Correlation between shots against and save percentage: {correlation:.3f}, p={p_value:.6f}")

    return shots_clean, save_pctg_clean

shots_clean, save_pctg_clean = check_shots_vs_save_pctg()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(shots_clean, save_pctg_clean, alpha=0.15, s=15)
plt.xlabel("Shots against")
plt.ylabel("Save percentage")
plt.title("Shots against vs. save percentage, per goalie start")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def check_shots_vs_save_pctg_full_games(season=LAST_SEASON, min_games=15, pause=0.5, min_minutes=55):
    """
    Same check as before, but restricted to appearances where the goalie
    played close to a full game (toi >= min_minutes), to rule out the
    'pulled after a bad start' confound, where a poor performance causes
    both the low shot count and the low save percentage, rather than
    low shots causing worse performance.
    """
    all_save_pctg = []
    all_shots_against = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)

        # toi comes back as a string like "58:42", convert to minutes
        toi_minutes = log["toi"].apply(lambda t: int(t.split(":")[0]) + int(t.split(":")[1]) / 60 if pd.notna(t) else None)
        full_games = log[toi_minutes >= min_minutes]

        all_save_pctg.extend(full_games["savePctg"].tolist())
        all_shots_against.extend(full_games["shotsAgainst"].tolist())

    valid_pairs = [(s, sv) for s, sv in zip(all_shots_against, all_save_pctg) if not pd.isna(sv)]
    shots_clean, save_pctg_clean = zip(*valid_pairs)

    correlation, p_value = stats.pearsonr(shots_clean, save_pctg_clean)
    print(f"Full-game appearances only: {len(shots_clean)} starts")
    print(f"Correlation between shots against and save percentage: {correlation:.3f}, p={p_value:.6f}")

    return shots_clean, save_pctg_clean

shots_full, save_pctg_full = check_shots_vs_save_pctg_full_games()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(shots_full, save_pctg_full, alpha=0.15, s=15)
plt.xlabel("Shots against")
plt.ylabel("Save percentage")
plt.title("Shots against vs. save percentage, per goalie start")
plt.grid(alpha=0.3)
plt.show()

## Completing the goalie opponent-shot-volume adjustment

Two pieces needed. First, how many shots does a given opponent
typically generate, this doesn't need a new pull, every goalie's
shotsAgainst in a game is exactly that game's opponent's shots
generated, so this can be built directly from goalie logs already
pulled.

Second, the actual relationship between shots faced and save
percentage, just confirmed at r=0.260 among full-game appearances,
fit as a simple linear relationship, so a projected shot volume can be
turned into a projected save percentage adjustment.

In [ ]:
def build_team_shot_generation_rates(season=LAST_SEASON, min_games=15, pause=0.5):
    """
    A team's shots generated per game equals the shotsAgainst faced by
    whichever goalie played them, so this reuses the same goalie pull
    already done elsewhere rather than hitting a new endpoint.
    """
    shots_by_team = {}

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)

        for opponent, shots in zip(log["opponentAbbrev"], log["shotsAgainst"]):
            shots_by_team.setdefault(opponent, []).append(shots)

    rates = {team: sum(values) / len(values) for team, values in shots_by_team.items()}
    return pd.DataFrame(rates.items(), columns=["team", "shots_generated_per_game"]).sort_values(
        "shots_generated_per_game", ascending=False
    ).reset_index(drop=True)

team_shot_rates = build_team_shot_generation_rates()
team_shot_rates.head(10)

In [ ]:
from scipy.stats import linregress

fit = linregress(shots_full, save_pctg_full)
print(f"save_pctg = {fit.intercept:.4f} + {fit.slope:.6f} * shots_against")
print(f"r-value: {fit.rvalue:.3f}, p-value: {fit.pvalue:.6f}")

def project_save_pctg(shots_against):
    """
    Projects expected save percentage given a shot volume, using the
    fitted relationship from full-game appearances. This is a league-
    wide average relationship, not specific to any one goalie, it
    adjusts around a goalie's own baseline, not in place of it.
    """
    return fit.intercept + fit.slope * shots_against

In [ ]:
league_avg_shots = team_shot_rates["shots_generated_per_game"].mean()

def adjust_goalie_for_opponent(player_id, opponent_team, season=LAST_SEASON):
    if opponent_team not in team_shot_rates["team"].values:
        return 0.0

    projected_shots = team_shot_rates.loc[
        team_shot_rates["team"] == opponent_team, "shots_generated_per_game"
    ].iloc[0]

    save_pctg_at_league_avg = project_save_pctg(league_avg_shots)
    save_pctg_at_projected = project_save_pctg(projected_shots)

    sv_weight = goalie_scoring.get("SV", 0)
    ga_weight = goalie_scoring.get("GA", 0)

    # volume effect, extra shots split into saves (SV weight) and
    # goals against (GA weight), using the league-average save rate
    shots_delta = projected_shots - league_avg_shots
    extra_saves_volume = shots_delta * save_pctg_at_league_avg
    extra_ga_volume = shots_delta * (1 - save_pctg_at_league_avg)

    # quality effect, the save-percentage shift itself, applied at the
    # projected shot level, also split the same way
    save_pctg_delta = save_pctg_at_projected - save_pctg_at_league_avg
    extra_saves_quality = save_pctg_delta * projected_shots
    extra_ga_quality = -save_pctg_delta * projected_shots

    adjustment = (
        (extra_saves_volume + extra_saves_quality) * sv_weight +
        (extra_ga_volume + extra_ga_quality) * ga_weight
    )
    return adjustment

In [ ]:
strongest_offense = team_shot_rates.iloc[0]["team"]
weakest_offense = team_shot_rates.iloc[-1]["team"]

adj_strong = adjust_goalie_for_opponent(goalie_id, strongest_offense)
adj_weak = adjust_goalie_for_opponent(goalie_id, weakest_offense)

print(f"{goalie_name} vs {strongest_offense} (high shot volume): {adj_strong:+.2f} fantasy points")
print(f"{goalie_name} vs {weakest_offense} (low shot volume): {adj_weak:+.2f} fantasy points")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(shots_full, save_pctg_full, alpha=0.15, s=15, label="Full-game appearances")

x_range = np.linspace(min(shots_full), max(shots_full), 100)
plt.plot(x_range, fit.intercept + fit.slope * x_range, color="red", linewidth=2,
          label=f"Fitted line (r={fit.rvalue:.3f})")

plt.xlabel("Shots against")
plt.ylabel("Save percentage")
plt.title("Shots against vs. save percentage, with fitted relationship")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(team_shot_rates["team"], team_shot_rates["shots_generated_per_game"], color="steelblue")
plt.xlabel("Shots generated per game")
plt.title("Team shot generation rates, league-wide")
plt.gca().invert_yaxis()  # highest at top
plt.grid(alpha=0.3, axis="x")
plt.show()

In [ ]:
adjustments = []
for team in team_shot_rates["team"]:
    adj = adjust_goalie_for_opponent(goalie_id, team)
    adjustments.append({"team": team, "adjustment": adj})

adjustments_df = pd.DataFrame(adjustments).sort_values("adjustment", ascending=False)

plt.figure(figsize=(10, 6))
colors = ["green" if a > 0 else "red" for a in adjustments_df["adjustment"]]
plt.barh(adjustments_df["team"], adjustments_df["adjustment"], color=colors)
plt.xlabel("Fantasy point adjustment")
plt.title(f"{goalie_name}, opponent shot-volume adjustment by team")
plt.axvline(0, color="black", linewidth=0.8)
plt.gca().invert_yaxis()
plt.grid(alpha=0.3, axis="x")
plt.show()

In [ ]:
def plot_shot_adjustment_for_goalie(name_query):
    """
    Looks up a goalie by name and plots their opponent shot-volume
    adjustment across every team in the league.
    """
    player_id = lookup_player_id(name_query)
    player_name = all_players[all_players["player_id"] == player_id].iloc[0]["name"]

    adjustments = []
    for team in team_shot_rates["team"]:
        adj = adjust_goalie_for_opponent(player_id, team)
        adjustments.append({"team": team, "adjustment": adj})

    adjustments_df = pd.DataFrame(adjustments).sort_values("adjustment", ascending=False)

    plt.figure(figsize=(10, 6))
    colors = ["green" if a > 0 else "red" for a in adjustments_df["adjustment"]]
    plt.barh(adjustments_df["team"], adjustments_df["adjustment"], color=colors)
    plt.xlabel("Fantasy point adjustment")
    plt.title(f"{player_name}, opponent shot-volume adjustment by team")
    plt.axvline(0, color="black", linewidth=0.8)
    plt.gca().invert_yaxis()
    plt.grid(alpha=0.3, axis="x")
    plt.show()

    return adjustments_df

vasilevskiy_adj = plot_shot_adjustment_for_goalie("Vasilevskiy")
swayman_adj = plot_shot_adjustment_for_goalie("Swayman")
wedgewood_adj = plot_shot_adjustment_for_goalie("Wedgewood")

In [ ]:
def project_final_estimate(player_id, opponent_team, position="G"):
    estimates = get_player_estimates(player_id, position)
    adjustment = adjust_goalie_for_opponent(player_id, opponent_team)
    return estimates["season_mu"] + adjustment

for name, pid in [("Brossoit", goalie_id), ("Vasilevskiy", lookup_player_id("Vasilevskiy")), ("Swayman", lookup_player_id("Swayman"))]:
    final_col = project_final_estimate(pid, "COL")
    final_chi = project_final_estimate(pid, "CHI")
    print(f"{name}: vs COL = {final_col:.2f}, vs CHI = {final_chi:.2f}")

## Note on the opponent shot-volume adjustment

The adjustment itself, roughly 0.4 fantasy points between the highest
and lowest shot-volume opponents in this test, is deliberately the
same shift applied to every goalie. It comes from a league-wide fitted
relationship (r=0.260) pooled across all goalies, since no single
goalie's season provides enough data to fit their own personal
version of this relationship reliably. What differs goalie to goalie
is the baseline it's added to, each goalie's own season-level mu, not
the size of the shift. A goalie-specific version of this adjustment
isn't ruled out, but would need either much more data per goalie or a
partial-pooling approach (each goalie's own trend blended with the
league-wide one, similar in spirit to the shrinkage estimator already
used for mu/sigma), noted in future_directions.

## Goalie-specific shot-volume effects, done honestly

A single goalie's own data isn't enough to reliably fit their own
slope, too few games per shot-volume bucket. Rather than either
pretending it's fine (overfitting to noise) or refusing to try at all,
this blends each goalie's own trend with the league-wide trend already
fitted, more weight on their own data as more of it accumulates, same
shrinkage logic already used for mu and sigma, just applied here to a
slope instead of a mean.

In [ ]:
def plot_goalie_vs_league(name_query, season=LAST_SEASON):
    player_id = lookup_player_id(name_query)
    player_name = all_players[all_players["player_id"] == player_id].iloc[0]["name"]

    log = get_game_log(player_id, season=season, force_refresh=False)
    log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)

    toi_minutes = log["toi"].apply(lambda t: int(t.split(":")[0]) + int(t.split(":")[1]) / 60 if pd.notna(t) else None)
    full_games = log[toi_minutes >= 55]

    plt.figure(figsize=(8, 5))
    plt.scatter(shots_full, save_pctg_full, alpha=0.06, s=15, color="gray", label="League, all goalies")
    plt.scatter(full_games["shotsAgainst"], full_games["savePctg"], alpha=0.8, s=40, color="red", label=player_name)

    x_range = np.linspace(min(shots_full), max(shots_full), 100)
    plt.plot(x_range, fit.intercept + fit.slope * x_range, color="black", linewidth=1.5, label="League fit")

    plt.xlabel("Shots against")
    plt.ylabel("Save percentage")
    plt.title(f"{player_name} vs. league, shots against and save percentage")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    return full_games

brossoit_games = plot_goalie_vs_league("Brossoit")
vasilevskiy_games = plot_goalie_vs_league("Vasilevskiy")
swayman_games = plot_goalie_vs_league("Swayman")
wedgewood_games = plot_goalie_vs_league("Wedgewood")

In [ ]:
def fit_goalie_specific_slope(shots, save_pctgs, league_intercept, league_slope, k=200):
    """
    Blends a goalie's own fitted slope/intercept with the league-wide
    fit, weighted by how many of their own games are available. k
    controls how much data it takes before a goalie's own trend starts
    to dominate the league trend, deliberately large here, since a
    slope is a much noisier thing to estimate than a mean, needs more
    evidence before trusting an individual deviation from the league.
    """
    n = len(shots)
    if n < 5:
        # not enough games to say anything about this goalie individually
        return league_intercept, league_slope

    own_fit = linregress(shots, save_pctgs)

    blended_slope = (n * own_fit.slope + k * league_slope) / (n + k)
    blended_intercept = (n * own_fit.intercept + k * league_intercept) / (n + k)

    return blended_intercept, blended_slope

def plot_goalie_specific_fit(name_query, games_df):
    player_name = all_players[all_players["player_id"] == lookup_player_id(name_query)].iloc[0]["name"]

    blended_intercept, blended_slope = fit_goalie_specific_slope(
        games_df["shotsAgainst"], games_df["savePctg"], fit.intercept, fit.slope
    )

    x_range = np.linspace(min(shots_full), max(shots_full), 100)

    plt.figure(figsize=(8, 5))
    plt.scatter(games_df["shotsAgainst"], games_df["savePctg"], alpha=0.6, s=30, label=f"{player_name}'s games")
    plt.plot(x_range, fit.intercept + fit.slope * x_range, color="gray", linestyle="--", label="League fit")
    plt.plot(x_range, blended_intercept + blended_slope * x_range, color="red", linewidth=2, label=f"{player_name}, blended fit")

    plt.xlabel("Shots against")
    plt.ylabel("Save percentage")
    plt.title(f"{player_name}, own trend blended with league trend ({len(games_df)} games)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    return blended_intercept, blended_slope

for name, games in [("Brossoit", brossoit_games), ("Vasilevskiy", vasilevskiy_games),
                     ("Swayman", swayman_games), ("Wedgewood", wedgewood_games)]:
    intercept, slope = plot_goalie_specific_fit(name, games)
    print(f"{name}: blended slope = {slope:.6f} (league slope = {fit.slope:.6f})")

In [ ]:
def compute_all_goalie_slopes(season=LAST_SEASON, min_games=15, pause=0.5):
    """
    Runs the partial-pooling slope fit for every goalie with a
    reasonable sample size, so the Vasilevskiy/Swayman/Wedgewood
    pattern can be checked against the full goalie population, not
    just three hand-picked names.
    """
    results = []

    for _, goalie in all_players[all_players["position"] == "G"].iterrows():
        log = get_game_log(goalie["player_id"], season=season, force_refresh=False)
        time.sleep(pause)

        if len(log) < min_games:
            continue

        log = clean_goalie_log(log, GOALIE_NUMERIC_COLUMNS)
        toi_minutes = log["toi"].apply(lambda t: int(t.split(":")[0]) + int(t.split(":")[1]) / 60 if pd.notna(t) else None)
        full_games = log[toi_minutes >= 55]

        if len(full_games) < 5:
            continue

        blended_intercept, blended_slope = fit_goalie_specific_slope(
            full_games["shotsAgainst"], full_games["savePctg"], fit.intercept, fit.slope
        )

        results.append({
            "goalie": goalie["name"],
            "games": len(full_games),
            "avg_save_pctg": full_games["savePctg"].mean(),
            "blended_slope": blended_slope,
        })

    return pd.DataFrame(results).sort_values("blended_slope").reset_index(drop=True)

goalie_slopes = compute_all_goalie_slopes()
goalie_slopes

In [ ]:
correlation, p_value = stats.pearsonr(goalie_slopes["avg_save_pctg"], goalie_slopes["blended_slope"])
print(f"Correlation between a goalie's average save % and their blended slope: {correlation:.3f}, p={p_value:.4f}")

plt.figure(figsize=(8, 5))
plt.scatter(goalie_slopes["avg_save_pctg"], goalie_slopes["blended_slope"], alpha=0.6)
plt.axhline(fit.slope, color="gray", linestyle="--", label="League slope")
plt.xlabel("Goalie's average save percentage")
plt.ylabel("Blended slope")
plt.title("Does a better goalie have a flatter shot-volume slope?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Does a goalie's overall quality predict their shot-volume slope?

Across 67 goalies with sufficient full-game samples, blended slope
correlates negatively with average save percentage, r=-0.284,
p=0.0199. Better goalies do tend to show a flatter relationship
between shots faced and save percentage than the league average,
consistent with the idea that elite goaltending is less dependent on
shot volume for effectiveness. The relationship is real but moderate,
r² is under 0.09, meaning save percentage alone explains a small
fraction of the variation in slope, plenty of exceptions exist in both
directions. Worth treating as a real, if modest, tendency, not a
strong individual predictor.

## Section 10 recap, goalie adjustments, tested rather than assumed

Two adjustments investigated, both handled with the same standard,
measure before building, don't assume an effect just because it's
intuitive.

Back-to-back starts: tested across 142 goalies, 53 back-to-back games
total, effect not statistically distinguishable from zero (p=0.560).
Shelved, not implemented, documented honestly as insufficient evidence
rather than a confirmed non-effect, revisit once more seasons of
pooled data exist.

Opponent shot volume: tested two versions, opponent goals-for (too
weak, r=-0.137) and shots-against vs. save percentage (r=0.260 among
full-game appearances, survived checks for two plausible confounds,
the small-denominator arithmetic effect and goalies pulled early after
a bad start). Built into the model as a league-wide pooled adjustment
applied on top of each goalie's own baseline mu.

Explored further, whether individual goalies deviate from the
league-wide slope, using partial pooling, each goalie's own trend
blended with the league trend, weighted by how many of their own games
exist. Found a real but modest relationship, better goalies tend to
have flatter slopes (r=-0.284, p=0.0199, n=67 goalies), worth noting as
a real tendency, not a strong individual predictor, and worth a
caveat about some circularity in how it was measured.

This closes the goalie side of Section 10. Skater opponent-strength
(mirroring this using opponent goals-against rather than goals-for) is
the next priority once the base model is stable, per future_directions.